# MS COCO Hybrid KD: final closeout

Notebook CPU này đóng nhánh Hybrid sau khi audit và paired bootstrap hoàn tất.
Nó tạo final lock cùng ba bảng ×100 dùng trực tiếp trong báo cáo khoa học.

## Input bắt buộc

1. Output `HYB_Audit`;
2. output `HYB_Compare`.

Runtime: **CPU / Accelerator None / Internet Off**. Không cần gắn các run seed,
MS COCO, checkpoint hoặc `pycocoevalcap`.

In [1]:
import csv
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "HYB_Closeout"
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
OUTPUT_ROOT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(temp, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (name, [str(path) for path in matches])
    return matches[0]

def verify_package(result_name):
    result_path = exactly_one(result_name)
    manifest_path = result_path.parent / "MANIFEST.json"
    manifest = read_json(manifest_path)
    assert manifest["status"] == "complete"
    for artifact in manifest["artifacts"]:
        path = result_path.parent / artifact["relative_path"]
        assert path.is_file(), path
        assert path.stat().st_size == int(artifact["size_bytes"])
        assert sha256(path) == artifact["sha256"]
    return result_path, read_json(result_path)

audit_path, audit = verify_package("HYB_AUDIT.json")
compare_path, compare = verify_package("HYB_COMPARE.json")
assert audit["status"] == "passed"
assert audit["decision"] == "AGGREGATE_PASSED_COMPARISON_REQUIRED"
assert compare["status"] == "complete"
assert compare["decision"] == "RETAIN_SEQUENCE_KD_REPORT_HYBRID_ABLATION"
assert compare["audit_sha256"] == sha256(audit_path)
assert audit["test_set_used"] is compare["test_set_used"] is False
assert compare["held_out_test_may_be_opened_for_hybrid"] is False
assert compare["onnx_or_int8_authorized_for_hybrid"] is False

summary = audit["summary"]
main_rows = []
for method, key in (("S1 baseline", "baseline"), ("Hybrid KD", "hybrid")):
    row = {"method": method}
    for metric in ("BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D"):
        row[f"{metric}_mean_x100"] = round(summary[metric][f"{key}_mean"] * 100, 2)
        row[f"{metric}_sample_sd_x100"] = round(summary[metric][f"{key}_sample_sd"] * 100, 2)
    row["CE_mean_raw"] = round(summary["CE"][f"{key}_mean"], 6)
    row["CE_sample_sd_raw"] = round(summary["CE"][f"{key}_sample_sd"], 6)
    main_rows.append(row)

main_csv = OUTPUT_ROOT / "HYB_MAIN_TABLE_X100.csv"
with main_csv.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(main_rows[0]))
    writer.writeheader()
    writer.writerows(main_rows)

comparison = audit["descriptive_comparison"]
method_rows = [
    {"method": "S1 baseline", "CIDEr_D_mean_x100": round(summary["CIDEr-D"]["baseline_mean"] * 100, 2)},
    {"method": "Dense Mapped Logit-KD", "CIDEr_D_mean_x100": comparison["dense_mapped_CIDEr_D_x100"]},
    {"method": "Hybrid KD", "CIDEr_D_mean_x100": comparison["hybrid_CIDEr_D_x100"]},
    {"method": "Sequence KD", "CIDEr_D_mean_x100": comparison["sequence_KD_CIDEr_D_x100"]},
]
method_csv = OUTPUT_ROOT / "HYB_METHOD_CIDER_X100.csv"
with method_csv.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(method_rows[0]))
    writer.writeheader()
    writer.writerows(method_rows)

bootstrap_rows = []
for comparison_name, seeds in compare["comparisons"].items():
    for seed, metrics in seeds.items():
        result = metrics["CIDEr-D"]
        bootstrap_rows.append({
            "comparison": comparison_name,
            "seed": int(seed),
            "delta_CIDEr_D_x100": round(result["mean_delta"] * 100, 2),
            "CI95_lower_x100": round(result["lower_95"] * 100, 2),
            "CI95_upper_x100": round(result["upper_95"] * 100, 2),
            "positive_lower_95": bool(result["positive_lower_95"]),
        })
bootstrap_csv = OUTPUT_ROOT / "HYB_BOOTSTRAP_CIDER_X100.csv"
with bootstrap_csv.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(bootstrap_rows[0]))
    writer.writeheader()
    writer.writerows(bootstrap_rows)

lock = {
    "schema_version": "hyb_final_lock_v1",
    "created_at_utc": now(),
    "status": "closed",
    "experiment_id": EXPERIMENT_ID,
    "method": "Hybrid Sequence and Dense-Mapped Logit Knowledge Distillation",
    "scientific_outcome": "POSITIVE_VS_BASELINE_NO_INCREMENTAL_GAIN_VS_SEQUENCE_KD",
    "role_in_report": "ablation",
    "selected_primary_method": "Sequence KD T1->S1",
    "hybrid_three_seed_CIDEr_D_x100": comparison["hybrid_CIDEr_D_x100"],
    "sequence_three_seed_CIDEr_D_x100": comparison["sequence_KD_CIDEr_D_x100"],
    "dense_mapped_three_seed_CIDEr_D_x100": comparison["dense_mapped_CIDEr_D_x100"],
    "hybrid_vs_baseline_CIDEr_D_delta_x100": round(summary["CIDEr-D"]["paired_delta_mean"] * 100, 2),
    "paired_bootstrap_gate_vs_sequence": compare["gates"]["hybrid_beats_sequence_positive_CIDEr_D_CI"],
    "decision": "RETAIN_SEQUENCE_KD_REPORT_HYBRID_ABLATION",
    "additional_training_authorized": False,
    "held_out_test_may_be_opened_for_hybrid": False,
    "onnx_or_int8_authorized_for_hybrid": False,
    "android_deployment_authorized_for_hybrid": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "source_sha256": {
        "audit": sha256(audit_path),
        "compare": sha256(compare_path),
    },
    "next": "Integrate the three-seed and paired-bootstrap tables into the scientific report.",
}
lock_path = OUTPUT_ROOT / "HYB_FINAL_LOCK.json"
write_json(lock_path, lock)

report_path = OUTPUT_ROOT / "REPORT.md"
report_path.write_text(
    "# MS COCO 2014 — Hybrid KD closeout\n\n"
    "## Kết luận\n\n"
    "- Ba seed và toàn vẹn hiện vật: **PASSED**.\n"
    f"- CIDEr-D Hybrid: **{comparison['hybrid_CIDEr_D_x100']:.2f}**.\n"
    f"- CIDEr-D Sequence KD: **{comparison['sequence_KD_CIDEr_D_x100']:.2f}**.\n"
    f"- CIDEr-D Dense Mapped Logit-KD: **{comparison['dense_mapped_CIDEr_D_x100']:.2f}**.\n"
    f"- Δ Hybrid so với baseline: **{summary['CIDEr-D']['paired_delta_mean'] * 100:+.2f}**.\n"
    "- Paired-bootstrap Hybrid > Sequence KD: **0/3 seed**.\n"
    "- Quyết định: giữ Sequence KD làm phương pháp chính; Hybrid được báo cáo như ablation.\n"
    "- Không mở Held-out Test, không xuất ONNX/INT8 và không triển khai Hybrid lên Android.\n",
    encoding="utf-8",
)

artifacts = [lock_path, main_csv, method_csv, bootstrap_csv, report_path]
write_json(OUTPUT_ROOT / "MANIFEST.json", {
    "schema_version": "hyb_closeout_manifest_v1",
    "status": "complete",
    "artifacts": [
        {"relative_path": path.name, "size_bytes": path.stat().st_size, "sha256": sha256(path)}
        for path in artifacts
    ],
    "test_set_used": False,
})

print("Closeout status           : COMPLETE")
print("Scientific outcome        :", lock["scientific_outcome"])
print("Selected primary method   :", lock["selected_primary_method"])
print("Hybrid role               :", lock["role_in_report"])
print("Held-out Test             : SEALED")
print("ONNX / INT8 / Android     : NOT AUTHORIZED")

Closeout status           : COMPLETE
Scientific outcome        : POSITIVE_VS_BASELINE_NO_INCREMENTAL_GAIN_VS_SEQUENCE_KD
Selected primary method   : Sequence KD T1->S1
Hybrid role               : ablation
Held-out Test             : SEALED
ONNX / INT8 / Android     : NOT AUTHORIZED


## Sau closeout

Không chạy thêm Hybrid trên Test, không xuất ONNX/INT8 và không thay mô hình
Android. Bước còn lại là đưa bảng ba seed và paired bootstrap vào báo cáo.